# 00 - Quickstart: Build a RAG in 30 lines

**Goal**: Feel the "compositional" nature of LangChain - turning a PDF into a
question-answering system takes just 5 components chained with `|`.

**Document**: `data/uploads/wells_fargo.pdf` (Wells Fargo Q4 2025 earnings release)

**Estimated time**: 10 minutes + a few API calls (~$0.01)

---

> Before you start: download the Wells Fargo Q4 2025 earnings PDF to
> `data/uploads/wells_fargo.pdf`:
>
> https://www.wellsfargo.com/assets/pdf/about/investor-relations/earnings/fourth-quarter-2025-earnings.pdf


## Step 1: Five imports is all you need

A complete RAG needs: **Loader + Splitter + Embeddings + VectorStore + LLM**.
LangChain provides production-ready implementations for each.


In [1]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Load OPENAI_API_KEY from .env
from dotenv import load_dotenv
load_dotenv()

print("All imports loaded.")

All imports loaded.


## Step 2: Load - turn the PDF into LangChain Documents

In [7]:
docs = PyMuPDFLoader("../data/uploads/wells_fargo.pdf").load()
print(f"Loaded {len(docs)} pages")
print(f"\nFirst 200 characters of page 1:")
print(docs[0].page_content)

Loaded 12 pages

First 200 characters of page 1:
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
  
News Release | January 14, 2026 
Wells Fargo Reports Fourth Quarter 2025 Net Income of 
$5.4 billion, or $1.62 per Diluted Share 
Net income, excluding a notable item, of $5.8 billion, or $1.76 per diluted 
share1 
Company-wide Financial Summary 
Selected Income Statement Data 
($ in millions except per share amounts) 
Quarter ended 
Dec 31, 
Dec 31, 
2025 
2024 
Total revenue 
$ 21,292 
20,378 
Noninterest expense 
13,726 
13,900 
Provision for credit losses2 
1,040 
1,095 
Net income 
5,361 
5,079 
Diluted earnings per common share 
1.62 
1.43 
Selected Balance Sheet Data 
($ in billions) 
Average loans 
Average deposits 
CET13 
$ 
955.8 
1,377.7 
10.6% 
906.4 
1,353.8 
11.1 
Performance Metrics 
ROE4 
12.3% 
11.7 
ROTCE5 
14.5 
13.9 
Operating Segments and Other Highlights 
Quarter 
Dec 31, 2025 
ended 
% Change from 
($ in billions) 
Dec 31, 
2025 
Sep 30,
Dec 31, 
2025 
2024 
Average lo

## Step 3: Split - cut into retrievable chunks

In [3]:
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=80)
chunks = splitter.split_documents(docs)
print(f"Split into {len(chunks)} chunks")
print(f"\nFirst chunk:\n{chunks[0].page_content[:200]}")

Split into 83 chunks

First chunk:
News Release | January 14, 2026 
Wells Fargo Reports Fourth Quarter 2025 Net Income of 
$5.4 billion, or $1.62 per Diluted Share 
Net income, excluding a notable item, of $5.8 billion, or $1.76 per di


## Step 4: Index - load into Chroma with OpenAI embeddings

In [4]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = Chroma.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

# Test
results = retriever.invoke("What was Wells Fargo's net income?")
print(f"Retrieved {len(results)} relevant chunks")
print(f"\nTop chunk:\n{results[0].page_content[:300]}")

Retrieved 4 relevant chunks

Top chunk:
News Release | January 14, 2026 
Wells Fargo Reports Fourth Quarter 2025 Net Income of 
$5.4 billion, or $1.62 per Diluted Share 
Net income, excluding a notable item, of $5.8 billion, or $1.76 per diluted 
share1 
Company-wide Financial Summary 
Selected Income Statement Data 
($ in millions except


## Step 5: Wire everything into a chain - the "feel" moment

The `|` operator is the heart of LangChain Expression Language (LCEL).
It lets you connect components like Lego bricks:

- `retriever` turns a query into a list of Documents
- `prompt` renders (context, question) into a prompt
- `llm` turns the prompt into a response
- `StrOutputParser` turns the response into a string

Reading the chain almost reads like English.


In [5]:
prompt = ChatPromptTemplate.from_template("""You are a financial analyst assistant.
Answer the question based ONLY on this context. If the answer isn't there, say so.

Context:
{context}

Question: {question}

Answer:""")

llm = ChatOpenAI(model="gpt-5-mini", temperature=1)

# This is the chain. It reads almost like English.
chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("Chain built. Run a few queries:")

Chain built. Run a few queries:


## Step 6: Ask questions - watch it work

In [6]:
questions = [
    "What was Wells Fargo's Q4 2025 net income?",
    "How did the Consumer Banking segment perform?",
    "What is the CET1 ratio?",
]

for q in questions:
    print(f"Q: {q}")
    print(f"A: {chain.invoke(q)}\n")
    print("-" * 60)

Q: What was Wells Fargo's Q4 2025 net income?
A: Wells Fargo's Q4 2025 net income was $5.4 billion.

------------------------------------------------------------
Q: How did the Consumer Banking segment perform?
A: The Consumer Banking and Lending segment provides diversified products for consumers and small businesses, and its revenue decreased 3% in 4Q25 versus 4Q24.

------------------------------------------------------------
Q: What is the CET1 ratio?
A: The CET1 ratio is 10.6% (as of December 31, 2025 — a preliminary estimate calculated under the Standardized Approach).

------------------------------------------------------------


## But pause - ask yourself these questions

Those 30 lines look perfect. You can ask any question, get a reasonable answer.
**But if you were the Capco interviewer, here's what they'd ask:**

| Question | Can you answer? |
|---|---|
| Is `chunk_size=500` optimal? How do you know? | ? |
| What happened to the 8 financial tables in the PDF? (Hint: mostly thrown away) | ? |
| When the system answers wrong, how do you debug? | ? |
| Are the answers really from the PDF, or is the LLM making things up? How do you test? | ? |
| What happens to performance when you go from 1 doc to 100? | ? |
| Once it's deployed, how do you monitor latency, cost, quality? | ? |

**These are exactly the questions the next 6 notebooks answer.**

| Notebook | Topic | Question it answers |
|---|---|---|
| 01_parsing | Real PDF parsing (tables + images + VLM caption) | "What happened to the tables?" |
| 02_chunking | Three chunking strategies compared | "Is chunk_size=500 optimal?" |
| 03_retrieval | Vector + BM25 + RRF hybrid | "Why is hybrid retrieval better?" |
| 04_generation | LangGraph orchestration + citations | "How do I debug?" |
| 05_evaluation | Ragas + 30-question golden set | "Are the answers true? How do I test?" |
| 06_observability | LangSmith + FastAPI deployment | "How do I monitor in production?" |

Ready? Open `01_parsing.ipynb`.
